# Module 1: Build the Graph

You are going to hand five hotel FAQ documents to an LLM and watch it turn unstructured
text into a typed knowledge graph. The five hotels you build are **not** in the graph you
just restored, so they join it permanently and you query them for the rest of the day.

## What this module does

| Step | What happens |
|------|--------------|
| Extraction | `SimpleKGPipeline` reads five documents and writes `Hotel`, `Room`, `Amenity`, `Policy`, and `Service` nodes |
| The pinned schema | The same five documents, extracted without a pinned schema, produce labels that do not agree with each other |
| Indexes | The vector index and the full-text index are created against the vectors your extraction just wrote |
| Verification | Every later module's contract is checked before you move on |

## The trade, said out loud

You extract five documents. The rest of the corpus arrived prebuilt in the dump, under this
same pinned schema, because extracting all of it takes hours and you would spend the workshop
watching a progress bar. Your five are real, they are yours, and they stay.

## Setup

The shared `workshop` package lives at the `notebooks/` root. The build machinery lives beside
`prepare_graph.py` in the Module 2 folder, which is where it was first written; both are added
to the path here.

In [ ]:
import os
import sys

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

# The shared `workshop` package at the notebooks/ root.
_notebooks_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _notebooks_root not in sys.path:
    sys.path.insert(0, _notebooks_root)

# graph_builder.py and graph_config.py, which build and verify the graph.
_build_machinery = os.path.join(_notebooks_root, "02-vector-rag-hallucinates")
if _build_machinery not in sys.path:
    sys.path.insert(0, _build_machinery)

In [ ]:
# Bedrock needs a region, and botocore reads only AWS_DEFAULT_REGION, never
# AWS_REGION. This sets both from one resolved value so that clients built
# without an explicit region_name land in the workshop's region instead of
# whatever the active AWS profile happens to configure.
from workshop.aws_region import configure_aws_region

configure_aws_region()

import boto3

identity = boto3.client("sts").get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")
print(f"Region: {boto3.session.Session().region_name}")
print("✅ AWS credentials configured")

## What is already in the graph

The dump you restored holds the corpus minus five documents. Look at it before you add anything,
so the numbers at the end of this notebook mean something.

In [ ]:
from graph_builder import connect, count_documents
from workshop.graph_connection import graph_database, require_neo4j_env

require_neo4j_env()

driver = connect()
with driver.session(database=graph_database()) as session:
    hotels_before = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]
documents_before = count_documents(driver)
driver.close()

print(f"Database: {graph_database()}")
print(f"Documents already loaded: {documents_before}")
print(f"Hotels already loaded:    {hotels_before}")

## The five documents that were held back

These five were kept out of the dump on purpose. No other file in this workshop names their
cities, and each of those cities still has its `-001` hotel in the graph, so nothing you build
here collides with anything already loaded.

That is the whole reason there is no cleanup step in this module. Your work is not a temporary
demonstration that gets deleted before Module 2; it is data the later modules query.

In [ ]:
from held_out_documents import HELD_OUT_DOCUMENTS, extract_held_out

paths = extract_held_out()
for path in paths:
    print(f"  {path.name}  ({path.stat().st_size:,} bytes)")

print(f"\n--- {paths[0].name}, first 400 characters ---")
print(paths[0].read_text(encoding="utf-8")[:400])

## Why the schema is pinned

`SimpleKGPipeline` will happily extract without a schema. Left to itself the LLM invents a fresh
vocabulary for every chunk: one document yields an `Address` node, the next puts the address on a
`Location`, a third calls it a `City`. Each result looks reasonable on its own. Together they are
unqueryable, because no single Cypher pattern matches all of them.

Every later module depends on these labels being stable. Module 3's retrieval tool promises the
agent that a hotel has `name`, `address`, and `guest_rating` on the node itself. That promise is
only keepable if extraction was constrained when the data was written.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA, OFF_SCHEMA_LABELS

print("Node types the extraction is allowed to produce:")
for node_type in GRAPH_SCHEMA["node_types"]:
    properties = ", ".join(p["name"] for p in node_type.get("properties", []))
    print(f"  :{node_type['label']:<9} {properties}")

print("\nRelationships it is allowed to produce:")
for start, rel, end in GRAPH_SCHEMA["patterns"]:
    print(f"  (:{start})-[:{rel}]->(:{end})")

print("\nLabels an unpinned run invents instead, which the build treats as a failure:")
print(f"  {', '.join(OFF_SCHEMA_LABELS)}")

### See it drift (optional)

This cell runs the extraction on **one** document with no schema pinned, into a throwaway
database-free view of what comes back, and prints the labels it produced. Run it if you want to
see the problem rather than read about it. Skip it if you would rather not spend the tokens; the
build below does not depend on it.

In [ ]:
# Optional. Extracts one document with no schema and reports what it invented.
RUN_UNPINNED_DEMO = False

if RUN_UNPINNED_DEMO:
    from graph_builder import clear_document, session, snapshot_chunk_ids
    from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import (
        FixedSizeSplitter,
    )
    from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

    from graph_config import CHUNK_OVERLAP, CHUNK_SIZE
    from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM

    sample = paths[0]
    driver = connect()
    try:
        baseline = snapshot_chunk_ids(driver)
        unpinned = SimpleKGPipeline(
            llm=BedrockLLM(),
            driver=driver,
            embedder=BedrockEmbeddings(),
            schema=None,  # the whole point
            text_splitter=FixedSizeSplitter(
                chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
            ),
            from_pdf=False,
            perform_entity_resolution=False,
        )
        await unpinned.run_async(
            file_path=sample.name,
            text=sample.read_text(encoding="utf-8"),
        )
        new_chunks = list(snapshot_chunk_ids(driver) - baseline)
        with session(driver) as neo4j_session:
            invented = neo4j_session.run(
                """
                MATCH (c:Chunk)<-[:FROM_CHUNK]-(n)
                WHERE elementId(c) IN $ids
                UNWIND [l IN labels(n) WHERE NOT l STARTS WITH '__'] AS label
                RETURN label, count(*) AS count ORDER BY count DESC
                """,
                ids=new_chunks,
            ).values()
        print(f"Unpinned extraction of {sample.name} produced: {invented}")
    finally:
        # Removed before the real build, so the drift demo cannot pollute it.
        clear_document(driver, sample.name)
        driver.close()
else:
    print("Skipped. Set RUN_UNPINNED_DEMO = True to watch the labels drift.")

## Build

`run_additive_build` extracts the five documents into the graph you already have. It never wipes.
The only thing it deletes is a previous copy of these same five documents, which is what makes
this cell safe to re-run if a call to Bedrock gets throttled.

It then creates the vector and full-text indexes and checks every contract the later modules
depend on before reporting success.

In [ ]:
from graph_builder import run_additive_build

exit_code = await run_additive_build(paths, "Module 1: building your five hotels")

if exit_code != 0:
    raise RuntimeError(
        "The build did not finish cleanly. Read the output above, then re-run "
        "this cell; it clears only your five documents before retrying."
    )

## Query what you built

Your five hotels are ordinary nodes in the same graph as everything that arrived in the dump.
There is no marker separating them and no reason for one.

In [ ]:
driver = connect()
with driver.session(database=graph_database()) as session:
    print("The hotels you just extracted:\n")
    for record in session.run(
        """
        MATCH (d:Document)<-[:FROM_DOCUMENT]-(:Chunk)<-[:FROM_CHUNK]-(h:Hotel)
        WHERE d.source_filename IN $filenames
        OPTIONAL MATCH (h)-[:OFFERS_AMENITY]->(a:Amenity)
        RETURN h.name AS name, h.address AS address,
               h.guest_rating AS rating, count(DISTINCT a) AS amenities
        ORDER BY name
        """,
        filenames=list(HELD_OUT_DOCUMENTS),
    ):
        print(f"  {record['name']}")
        print(f"    {record['address']}")
        print(f"    rating {record['rating']}, {record['amenities']} amenities\n")

    hotels_after = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]

documents_after = count_documents(driver)
driver.close()

print(f"Documents: {documents_before} -> {documents_after}")
print(f"Hotels:    {hotels_before} -> {hotels_after}")
print(
    "\nEvery aggregation and multi-hop question from here on runs across the whole "
    "graph, yours included."
)

## Before you move on: what a Strands agent is

Module 2 builds two agents in its first few cells, so here is the vocabulary, just ahead of first use.

**`Agent`** is the loop. You give it a model, a system prompt, and a list of tools. It sends the
user's message to the model, and when the model asks to call a tool the `Agent` runs it, feeds the
result back, and repeats until the model answers instead of calling something.

```python
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-sonnet-5"),
    system_prompt="Answer only from tool results. If a tool returns nothing, say so.",
    tools=[search_hotels],
)
```

**Pin the model.** `BedrockModel` takes the same model ID you would pass to `boto3`. Pinning it
means the answer you get in Module 3 is the answer the facilitator got, which matters when the
whole point of a module is comparing two outputs.

**`@tool`** turns a Python function into something the model can call. The docstring is not
decoration: it is the description the model reads when deciding whether this tool is the right one.

```python
from strands import tool

@tool
def search_hotels(question: str) -> str:
    """Search hotel knowledge. Use for questions about specific hotels."""
    return retriever.search(question)
```

The thing to carry into Module 2: an agent is only as grounded as its tools. A tool that returns
plausible-looking text for a question it cannot actually answer produces a confident wrong answer,
and no system prompt reliably prevents that. Module 2 shows exactly this failure, then fixes it.

In [ ]:
from workshop.workshop_utils import lego_progress

lego_progress(1)